In [70]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import re
from faker import Faker
import random

In [3]:
def create_db_connection(host_name, user_name, user_password, db_name):
    connection = None
    try:
        connection = mysql.connector.connect(
            host=host_name,
            user=user_name,
            passwd=user_password,
            database = db_name
        )
        print("MySQL Database connection successful")
    except Error as err:
        print(f"Error: '{err}'")

    return connection


def execute_query(connection, query, values=None):
    cursor = connection.cursor()
    try:
        if values:
            if isinstance(values[0], (list, tuple)):  
                    cursor.executemany(query, values)
            else: 
                cursor.execute(query, values)
        else:
            cursor.execute(query)
        connection.commit()
        print("Query successful")
    except Error as err:
        print(f"Error: '{err}'")


## Fake data

In [67]:
file_path = "sendo_products.csv"  
df = pd.read_csv(file_path)
df_product = df[['productID', 'productName', 'productBrand', 'productPrice', 'productDescription']]

In [ ]:
# Init Faker
fake_E = Faker()
fake_V = Faker('vi_VN')

### Fake Product


In [ ]:
categories = [
    ('ASM01', 'Áo sơ mi', 'Nam'),
    ('AT01', 'Áo thun', 'Nam'),
    ('AK01', 'Áo khoác', 'Nam'),
    ('QJ01', 'Quần jean', 'Nam'),
    ('QS01', 'Quần short', 'Nam'),
    ('QD01', 'Quần đùi', 'Nam'),
    ('GTT01', 'Giày thể thao', 'Nam'),
    ('GT01', 'Giày tây', 'Nam'),
    ('GSD01', 'Giày sandal', 'Nam'),
    ('ASM02', 'Áo sơ mi', 'Nữ'),
    ('AT02', 'Áo thun', 'Nữ'),
    ('AK02', 'Áo khoác', 'Nữ'),
    ('QJ02', 'Quần jean', 'Nữ'),
    ('QS02', 'Quần short', 'Nữ'),
    ('QD02', 'Quần đùi', 'Nữ'),
    ('GTT02', 'Giày thể thao', 'Nữ'),
    ('GT02', 'Giày tây', 'Nữ'),
    ('GSD02', 'Giày sandal', 'Nữ'),
    ('ASM03', 'Áo sơ mi', 'Unisex'),
    ('AT03', 'Áo thun', 'Unisex'),
    ('AK03', 'Áo khoác', 'Unisex'),
    ('QJ03', 'Quần jean', 'Unisex'),
    ('QS03', 'Quần short', 'Unisex'),
    ('QD03', 'Quần đùi', 'Unisex'),
]

df_product.loc[:, 'categoryID'] = 'A'
for index, row in df_product.iterrows():
    product_name = row['productName'].lower()
    found = False
    
    for category_id, category_name, gender in categories:
        if category_name.lower() in product_name and gender.lower() in product_name:
                df_product.loc[index, 'categoryID'] = category_id
                found = True
                break
    

    if not found:
        for category_id, category_name, _ in categories:
            if category_name.lower() in product_name:
                df_product.loc[index, 'categoryID'] = re.sub(r'\d', '', category_id)  
                break

df_product.head()

In [ ]:
# Define sizes
sizes = ["S", "M", "L"]

# Duplicate rows with different sizes
expanded_rows = []
for _, row in df_product.iterrows():
    for size in sizes:
        new_row = row.copy()
        new_row["size"] = size
        new_row["productID"] = f"{row['productID']}{size}"  # Append size to productID
        expanded_rows.append(new_row)

df_product_new = pd.DataFrame(expanded_rows)

# replace Nan values with 'None'
df_product_new = df_product_new.fillna('None')

df_product_new.head(10)



### Fake Order


In [ ]:
# Num rows to fake
num_customers = 10

customers = [(f"KH{i+1:03d}", fake_V.name(), random.choice(['Male', 'Female', 'Other']),
              fake.date_of_birth(minimum_age=18, maximum_age=60),
              fake.phone_number(), fake.address()) for i in range(num_customers)]

customer_query = """
INSERT INTO Customer (CustomerID, Name, Sex, Date_of_birth, PhoneNumber, Address) VALUES (%s, %s, %s, %s, %s, %s)
"""

# Dữ liệu cho Product
categories = ['Áo', 'Quần', 'Giày', 'Váy']
products = [(f"P{i+1:02d}", fake.word(), random.choice(categories),
             random.choice(['S', 'M', 'L', 'XL']), random.uniform(100000, 1000000), random.randint(1, 100)) for i in range(num_products)]

product_query = """
INSERT INTO Product (ProductID, Name, Category, Size, Price, Inventory_quantity) VALUES (%s, %s, %s, %s, %s, %s)
"""

# Dữ liệu cho Staff
positions = ['Nhân viên bán hàng', 'Quản lý kho', 'Thu ngân']
staffs = [(f"NV{i+1:02d}", fake.name(), random.choice(positions), random.uniform(5000000, 20000000)) for i in range(num_staff)]

staff_query = """
INSERT INTO Staff (StaffID, Name, Position, Salary) VALUES (%s, %s, %s, %s)
"""

# Dữ liệu cho Orders
orders = [(f"DH{i+1:02d}", random.choice(customers)[0], random.choice(staffs)[0],
           fake.date_time_this_year(), random.uniform(200000, 2000000), None, random.choice(['Cash', 'Card', 'Momo'])) for i in range(num_orders)]

order_query = """
INSERT INTO Orders (OrderID, CustomerID, StaffID, DateTime, TotalPrice, DiscountID, Payment_method) VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

# Dữ liệu cho Shipment
shipments = [(f"GH{i+1:02d}", random.choice(orders)[0], random.choice(customers)[0],
              random.choice(['Đang giao hàng', 'Đã giao', 'Chờ xử lý']), fake.company(), fake.sentence()) for i in range(num_shipments)]

shipment_query = """
INSERT INTO Shipment (ShipID, OrderID, CustomerID, State, Shipper, Shipper_info) VALUES (%s, %s, %s, %s, %s, %s)
"""

# Dữ liệu cho Discount
discounts = [(f"DSC{i+1:02d}", fake.word(), random.randint(5, 50), fake.date_time_this_year(), fake.date_time_this_year()) for i in range(num_discounts)]

discount_query = """
INSERT INTO Discount (DiscountID, Name, Percentage, Start_time, End_time) VALUES (%s, %s, %s, %s, %s)
"""

# Dữ liệu cho OrderDetail (tránh trùng khóa chính)
order_detail_set = set()
order_details = []
while len(order_details) < num_order_details:
    order_id = random.choice(orders)[0]
    product_id = random.choice(products)[0]
    if (order_id, product_id) not in order_detail_set:
        order_detail_set.add((order_id, product_id))
        order_details.append((order_id, product_id, random.randint(1, 5)))

order_detail_query = """
INSERT INTO OrderDetail (OrderID, ProductID, Quantity) VALUES (%s, %s, %s)
"""


## Insert

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

# import data into Product
for index, row in df_product_new.iterrows():
    insert_query = """
    INSERT INTO Product (ProductID, Name, Brand, Price, Description, Category, Size)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """
       
    values = (
        row["productID"], row["productName"], row["productBrand"], row["productPrice"], row["productDescription"],
        row["categoryID"], row["size"]
    )
    
    execute_query(connection, insert_query, values)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

### Insert Platform

In [ ]:
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = """
INSERT INTO Platfrom (ID, Platform_Name, Platform_URL)
VALUES ('SENDO', 'Sendo', 'https://www.sendo.vn/'),
         ('TIKI', 'Tiki', 'https://tiki.vn/'),
         ('LAZADA', 'Lazada', 'https://www.lazada.vn/')  
"""
execute_query(connection, insert_query)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

### Insert Category


In [ ]:
# insert data into Category
connection =create_db_connection("localhost", "root", "031103", "ecommerce_shop")
execute_query(connection, "SET FOREIGN_KEY_CHECKS = 0;")

insert_query = """
INSERT INTO Category (ID, Name, Type)
    VALUES ('ASM01','Áo sơ mi', 'Nam'),
           ('AT01', 'Áo thun', 'Nam'),
           ('AK01', 'Áo khoác', 'Nam'),
           ('QJ01', 'Quần jean', 'Nam'),
           ('QS01', 'Quần short', 'Nam'),
           ('QD01', 'Quần đùi', 'Nam'),
           ('GTT01', 'Giày thể thao', 'Nam'),
           ('GT01', 'Giày tây', 'Nam'),
           ('GSD01', 'Giày sandal', 'Nam'),
           ('ASM02','Áo sơ mi', 'Nữ'),
           ('AT02', 'Áo thun', 'Nữ'),
           ('AK02', 'Áo khoác', 'Nữ'),
           ('QJ02', 'Quần jean', 'Nữ'),
           ('QS02', 'Quần short', 'Nữ'),
           ('QD02', 'Quần đùi', 'Nữ'),
           ('GTT02', 'Giày thể thao', 'Nữ'),
           ('GT02', 'Giày tây', 'Nữ'),
           ('GSD02', 'Giày sandal', 'Nữ'),
           ('ASM03','Áo sơ mi', 'Unisex'),
           ('AT03', 'Áo thun', 'Unisex'),
           ('AK03', 'Áo khoác', 'Unisex'),
           ('QJ03', 'Quần jean', 'Unisex'),
           ('QS03', 'Quần short', 'Unisex'),
           ('QD03', 'Quần đùi', 'Unisex')
"""
execute_query(connection, insert_query)

# Close connection
if connection:
    connection.close()
    print("Đã đóng kết nối MySQL")

MySQL Database connection successful
Query successful
Query successful
Đã đóng kết nối MySQL
